# Market Anomaly Detection

**Presented By:** Mohd Abuzar | **Company:** STARlab Capital | **Date:** 20th May, 2026

**Dataset:** [hourly MSFT stock data (2017-2026)](https://starlabcapital-my.sharepoint.com/:x:/p/haider/IQA_TaNlHafsSZLUF2z2GeYwAXMGNnXJwfG-3NcL09gvawI?e=nGiYT8)

## Sections
1. Dataset Overview
2. Missing Value Analysis
3. Data Quality Checks
4. Price Analysis (OHLC distributions, time series)
5. Bid-Ask Spread Analysis
6. VWAP Analysis
7. Return Distribution Analysis
8. Correlation Analysis
9. Stationarity Testing (ADF test)
10. Seasonality & Periodicity Patterns
11. Initial Outlier Detection
12. Summary of Key Findings

In [ ]:
# Notebook imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "reports" / "figures"

# Notebook configuration
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (14, 5),
        "figure.dpi": 100,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "font.size": 11,
        "lines.linewidth": 1.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

COLORS = {
    "price": "#2563eb",
    "volume": "#7c3aed",
    "anomaly": "#dc2626",
    "spread": "#059669",
    "normal": "#6b7280",
    "highlight": "#f59e0b",
}

plots_dir = PLOTS_DIR
plots_dir.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / "raw" / "msft_hourly(in).csv"
print("Setup complete. Libraries loaded.")

In [ ]:
# State check: preprocessing is the starting point
def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def require_columns(frame, cols, name="df"):
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise RuntimeError(f"{name} missing columns: {', '.join(missing)}")


require(CSV_PATH.exists(), f"CSV not found at {CSV_PATH}. Update CSV_PATH in config.")
print("Preprocessing notebook: start here. No prerequisites.")

---
## 1. Dataset Overview & Documentation

**Description:** Hourly OHLCV (Open, High, Low, Close, Volume) market data
for Microsoft Inc. (MSFT) stock, including bid/ask quotes and VWAP.

In [ ]:
# Load raw data
df_raw = pd.read_csv(CSV_PATH)

print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"\nColumns ({df_raw.shape[1]}):")
for i, col in enumerate(df_raw.columns, 1):
    dtype = df_raw[col].dtype
    non_null = df_raw[col].notna().sum()
    print(f"  {i:2d}. {col:25s}  {str(dtype):10s}  {non_null}/{len(df_raw)} non-null")

In [ ]:
# Parse datetime and sort
if "df_raw" not in globals():
    raise RuntimeError("df_raw not found. Run the load-data cell first.")
require_columns(
    df_raw, ["quote_datetime", "open", "high", "low", "close"], name="df_raw"
)

df = df_raw.copy()
df["quote_datetime"] = pd.to_datetime(df["quote_datetime"], errors="coerce")
df = df.sort_values("quote_datetime").reset_index(drop=True)

# Drop unnamed index if present
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("DATASET DOCUMENTATION")
print("Source:          STARlab Capital provided dataset")
print("Asset:           MSFT (Microsoft Inc.)")
print(f"Time Range:      {df['quote_datetime'].min()} to {df['quote_datetime'].max()}")
print(
    f"Duration:        ~{(df['quote_datetime'].max() - df['quote_datetime'].min()).days / 365:.1f} years"
)
print("Frequency:       Hourly (with half-hour bar at 16:00 close)")
print(f"Total Rows:      {len(df):,}")
print(f"Trading Days:    {df['quote_datetime'].dt.date.nunique():,}")
print("\nKey Variables:")
print("  Price:    open, high, low, close")
print("  Quotes:   bid, ask, mid")
print("  Volume:   trade_volume (NOT used for anomaly detection per task spec)")
print("  VWAP:     vwap (volume-weighted average price)")

---
## 2. Descriptive Statistics

In [ ]:
# Descriptive statistics for all numeric columns
if "df" not in globals():
    raise RuntimeError("df not found. Run the parsing cell first.")

print("\nDescriptive Statistics:")
df_numeric = df.select_dtypes(
    include=[np.number]
)  # filters numeric columns from df (irrespective of dtypes)
display_stats = df_numeric.describe().T
display_stats["missing"] = df_numeric.isnull().sum()
display_stats["missing_%"] = (df_numeric.isnull().sum() / len(df) * 100).round(2)
print("\n", display_stats.to_string())

---
## 3. Data Quality Checks

We check for:
- Invalid prices (negative, zero)
- OHLC consistency (high >= low, close in [low, high])
- Bid-ask consistency (bid <= ask)
- Duplicate timestamps
- Negative Volume
- Extreme price changes (> 20% in one bar)

In [ ]:
print("DATA QUALITY CHECKS & FIXES")
if "df" not in globals():
    raise RuntimeError("df not found. Run the parsing cell first.")
require_columns(
    df, ["open", "high", "low", "close", "bid", "ask", "quote_datetime"], name="df"
)
print(f"Starting rows: {len(df)}")

# Keep chronological order before return-based checks
df = df.sort_values("quote_datetime").reset_index(drop=True)

# Check 1: Non-positive OHLC -> drop rows
price_cols = ["open", "high", "low", "close"]
invalid_price_mask = (df[price_cols] <= 0).any(axis=1)
invalid_price_rows = int(invalid_price_mask.sum())
print(f"\n[Check 1] Rows with non-positive OHLC values: {invalid_price_rows}")
if invalid_price_rows > 0:
    df = df.loc[~invalid_price_mask].reset_index(drop=True)
    print(f"  [Fix] Dropped {invalid_price_rows} invalid price rows")
else:
    print("  [Fix] No action needed")

# Check 2a: High < Low -> swap
high_lt_low_mask = df["high"] < df["low"]
high_lt_low = int(high_lt_low_mask.sum())
print(f"\n[Check 2a] High < Low violations: {high_lt_low}")
if high_lt_low > 0:
    tmp_low = df.loc[high_lt_low_mask, "low"].copy()
    df.loc[high_lt_low_mask, "low"] = df.loc[high_lt_low_mask, "high"]
    df.loc[high_lt_low_mask, "high"] = tmp_low
    print(f"  [Fix] Swapped High/Low for {high_lt_low} rows")
else:
    print("  [Fix] No action needed")

# Check 2b: Close outside [Low, High] -> clip
close_outside_mask = (df["close"] < df["low"]) | (df["close"] > df["high"])
close_outside = int(close_outside_mask.sum())
print(f"\n[Check 2b] Close outside [Low, High]: {close_outside}")
if close_outside > 0:
    df.loc[close_outside_mask, "close"] = df.loc[close_outside_mask, "close"].clip(
        lower=df.loc[close_outside_mask, "low"],
        upper=df.loc[close_outside_mask, "high"],
    )
    print(f"  [Fix] Clipped Close into [Low, High] for {close_outside} rows")
else:
    print("  [Fix] No action needed")

# Check 2c: Open outside [Low, High] -> clip
open_outside_mask = (df["open"] < df["low"]) | (df["open"] > df["high"])
open_outside = int(open_outside_mask.sum())
print(f"\n[Check 2c] Open outside [Low, High]: {open_outside}")
if open_outside > 0:
    df.loc[open_outside_mask, "open"] = df.loc[open_outside_mask, "open"].clip(
        lower=df.loc[open_outside_mask, "low"],
        upper=df.loc[open_outside_mask, "high"],
    )
    print(f"  [Fix] Clipped Open into [Low, High] for {open_outside} rows")
else:
    print("  [Fix] No action needed")

# Check 3: Bid-Ask consistency -> set bid/ask to mid (or swap if mid absent)
if "bid" in df.columns and "ask" in df.columns:
    bid_gt_ask_mask = df["bid"] > df["ask"]
    bid_gt_ask = int(bid_gt_ask_mask.sum())
    print(f"\n[Check 3] Bid > Ask violations: {bid_gt_ask}")

    if bid_gt_ask > 0:
        if "mid" in df.columns:
            mid_vals = df.loc[bid_gt_ask_mask, "mid"]
            df.loc[bid_gt_ask_mask, "bid"] = mid_vals.values
            df.loc[bid_gt_ask_mask, "ask"] = mid_vals.values
            print(f"  [Fix] Set Bid/Ask to Mid for {bid_gt_ask} rows")
        else:
            tmp_ask = df.loc[bid_gt_ask_mask, "ask"].copy()
            df.loc[bid_gt_ask_mask, "ask"] = df.loc[bid_gt_ask_mask, "bid"]
            df.loc[bid_gt_ask_mask, "bid"] = tmp_ask
            print(f"  [Fix] Swapped Bid/Ask for {bid_gt_ask} rows")
    else:
        print("  [Fix] No action needed")

# Check 4: Duplicate timestamps -> keep first
dups = int(df["quote_datetime"].duplicated().sum())
print(f"\n[Check 4] Duplicate timestamps: {dups}")
if dups > 0:
    df = df.drop_duplicates(subset=["quote_datetime"], keep="first").reset_index(
        drop=True
    )
    print(f"  [Fix] Dropped {dups} duplicate rows")
else:
    print("  [Fix] No action needed")

# Check 5: Negative volume -> set to 0
if "trade_volume" in df.columns:
    neg_vol_mask = df["trade_volume"] < 0
    neg_vol = int(neg_vol_mask.sum())
    print(f"\n[Check 5] Negative volume entries: {neg_vol}")
    if neg_vol > 0:
        df.loc[neg_vol_mask, "trade_volume"] = 0
        print(f"  [Fix] Set {neg_vol} negative volume values to 0")
    else:
        print("  [Fix] No action needed")

# Check 6: Extreme single-bar returns -> drop rows
returns = df["close"].pct_change()
extreme_mask = returns.abs() > 0.20
extreme = int(extreme_mask.sum())
print(f"\n[Check 6] Extreme returns (>20% in 1 bar): {extreme}")
if extreme > 0:
    df = df.loc[~extreme_mask].reset_index(drop=True)
    print(f"  [Fix] Dropped {extreme} rows with extreme 1-bar returns")
else:
    print("  [Fix] No action needed")

print("\nCLEANING COMPLETE")
print(f"Final rows: {len(df)}")
if "bid" in df.columns and "ask" in df.columns:
    print(f"Remaining Bid > Ask violations: {(df['bid'] > df['ask']).sum()}")
print(f"Remaining duplicate timestamps: {df['quote_datetime'].duplicated().sum()}")

In [ ]:
df

In [ ]:
# Project Paths Helper

_cwd = Path().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA_DIR = PROJECT_ROOT / "data"
PLOTS_DIR = PROJECT_ROOT / "reports" / "figures"

# Ensure processed CSV exists (write it if missing)

processed_path = DATA_DIR / "processed" / "msft_hourly(in)_processed.csv"
if not processed_path.exists():
    df.to_csv(processed_path, index=False)
    print(f"Saved processed data to: {processed_path}")
else:
    print(f"Processed data already exists: {processed_path}")